# Molecular MD statistics from a simulation directory

Point `SIMULATION_DIR` at your LAMMPS run directory and run the cells. This notebook reads the input and text dump directly; no reports from the earlier notebooks are required.

All displacement and correlation statistics use **all molecules of each species**. There is no arbitrary subset of atoms. Molecules are identified from the first frame's bond graph and retain those identities throughout. Formula labels distinguish species, but do not distinguish isomers. This is a nonreactive, fixed-topology analysis.

The analysis uses metal units and a constant timestep. Set UNITS and TIMESTEP_PS explicitly below. These values take precedence and bypass input parsing and header inference. An input file is not required. Set either value to None only when you want file metadata to supply it; restart binary files are not parsed. Supported dump suffixes are `.lammpstrj` and `.dump`; explicit filenames can use other suffixes. Unique input/dump files are discovered recursively. If several match, choose the filenames explicitly instead of silently analyzing the wrong run.

After updating the package, restart the kernel and run all cells so imported function definitions match the notebook. Explicit UNITS and TIMESTEP_PS bypass input-script parsing.


Window selection uses **zero-based source dump frames**: `STARTFRAME` is inclusive and `ENDFRAME` exclusive. The defaults (`0`, `None`) use the entire loaded trajectory. With stride 2, `[3, 9)` selects source frames 4, 6, 8. These indices are not LAMMPS timesteps or rows after striding. Lag and diffusion-fit settings remain in ps.

Pair analysis now caches initial topology and reuses whole-molecule coordinates. Start with `PAIR_WORKERS=1`; try `2` or `4` only if benchmarking shows a benefit. `PAIR_BACKEND="reference"` with one worker retains the older per-frame graph reconstruction. Cached analysis assumes fixed connectivity and preserves molecular ID sets; atom-list order is fixed from the first frame. Frames remain chronological for unwrapping and pair lifetimes. A notebook can call process workers directly; standalone scripts must use an `if __name__ == "__main__":` guard.

In [ ]:
from pathlib import Path
from dataclasses import asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from MartiniFuncs.trajectory_analysis import analyze_run, molecular_window_statistics
from MartiniFuncs import md_statistics as md
from MartiniFuncs.pair_selection import PairCriteria

# Usually this is the only path to change.
SIMULATION_DIR = Path(r"C:/Users/nicho/Documents/Martini collab/uma_data/smaller_system/continued_run")
INPUT_FILE = None   # optional with explicit UNITS/TIMESTEP_PS; e.g. "infile.in"; required only if discovery is ambiguous
DUMP_FILE = None    # e.g. "output/equilibrate.lammpstrj"
DATA_FILE = None    # optional species/mass fallback; use the matching initial data file

TYPE_SYMBOLS = None  # optional {1: "C", 2: "H", ...} for missing dump elements
UNITS = "metal"  # explicit LAMMPS source units
TIMESTEP_PS = 0.0005  # ps per MD step; 0.5 fs

FRAME_STRIDE = 1
MAX_FRAMES = None   # set e.g. 200 for a quick preview; caps SOURCE frames read
BOND_SCALE = 1.3
TEMPERATURE_DOF = "3N-3"  # unconstrained atoms, 3 COM DOF removed; change for constraints
# Set TEMPERATURE_DOF=None to omit temperature rather than assume this convention.

COMPUTE_PAIRS = True  # optional CPD/dienophile screen; cached by default
PAIR_CRITERIA = PairCriteria()  # distance-only unless you explicitly add thresholds
PAIR_BACKEND = "cached"  # reuse initial molecular topology; "reference" rebuilds each frame
PAIR_WORKERS = 1  # try 2 or 4 for larger workloads; processes can be slower for small runs
PAIR_BATCH_SIZE = 32  # retained frames per task; at most 2*workers tasks in flight

STARTFRAME = 0     # inclusive, zero-based source dump frame
ENDFRAME = None    # exclusive source frame; None uses the end
MAX_LAG_PS = 100.0
DIFFUSION_FIT_PS = None  # e.g. (50, 100), only after inspecting molecular MSD
SAVE_PLOTS = False
PLOT_DIR = SIMULATION_DIR / "md_statistics_plots"


`MAX_FRAMES` limits the available source frames for a preview; it does not select equilibrated data. `STARTFRAME`/`ENDFRAME` select a range within those available frames. Defaults use all available frames; selected frame bounds and physical times are printed.

In [ ]:
result = analyze_run(
    SIMULATION_DIR, input_file=INPUT_FILE, dump_file=DUMP_FILE, data_file=DATA_FILE,
    frame_stride=FRAME_STRIDE, max_frames=MAX_FRAMES, bond_scale=BOND_SCALE,
    degrees_of_freedom=TEMPERATURE_DOF, compute_pairs=COMPUTE_PAIRS,
    pair_criteria=PAIR_CRITERIA, pair_backend=PAIR_BACKEND,
    pair_workers=PAIR_WORKERS, pair_batch_size=PAIR_BATCH_SIZE, type_symbols=TYPE_SYMBOLS, units=UNITS, timestep_ps=TIMESTEP_PS, progress=print,
)
frames = pd.DataFrame(result["frames"])
print("Files:", result["files"])
print("Units sources:", result["settings"]["units_sources"])
print("Timestep (ps):", result["settings"]["timestep"])
print(f"Read {result['frames_read']} frames; retained {len(frames)}")
print(f"Loaded time: {frames.time_ps.iloc[0]:g} to {frames.time_ps.iloc[-1]:g} ps")
if MAX_FRAMES is not None:
    print("SOURCE FRAME LIMIT ACTIVE: the full trajectory may extend beyond this preview.")
display(pd.Series(result["species"]).value_counts().rename_axis("formula").to_frame("molecules"))
for notice in result["notices"]:
    print(notice)

## Choose the analysis window

The gray shading in the histories below marks the selected interval. This choice excludes early data from molecular MSD and correlations; it is not an automatic equilibration determination.

In [ ]:
window = molecular_window_statistics(
    result, startframe=STARTFRAME, endframe=ENDFRAME,
    max_lag_ps=MAX_LAG_PS,
)

print("Source frame range [start, end):", window["frame_window"])
start, end = window["time_window_ps"]
print(f"Analyzing {start:g}-{end:g} ps; {window['frame_mask'].sum()} retained frames")
print("Molecular MSD: whole-molecule COM then TOR unwrapping; no drift or finite-size correction.")

def finish(fig, name):
    if SAVE_PLOTS:
        PLOT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(PLOT_DIR / f"{name}.png", dpi=170, bbox_inches="tight")
        fig.savefig(PLOT_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
columns = [c for c in ("temperature_K", "density_g_cm3", "volume_angstrom3", "candidate_pairs") if c in frames]
fig, axes = plt.subplots(len(columns), 1, figsize=(10, 2.6*len(columns)), sharex=True, squeeze=False, layout="constrained")
for ax, column in zip(axes[:, 0], columns):
    ax.plot(frames.time_ps, frames[column], lw=.7)
    ax.axvspan(start, end, color="gray", alpha=.15)
    ax.set_ylabel(column)
    ax.grid(alpha=.2)
axes[-1, 0].set_xlabel("Trajectory time (ps)")
finish(fig, "frame_histories")

## Molecular COM MSD and optional diffusion fits

Each curve averages over every molecule of that formula and every available time origin within the selected window. The second panel shows how many time origins support each lag; overlapping origins are correlated.

TOR unwrapping is enabled only for fully periodic orthorhombic boxes with a fixed center. It assumes displacements between saved source frames are smaller than half the box. That assumption cannot be proved from wrapped coordinates alone. Triclinic/shear diffusion requires a separate deformation/reference-frame treatment.

A fitted slope is not automatically a long-time diffusion coefficient. Inspect the curve and choose `DIFFUSION_FIT_PS` explicitly. Fits use MSD = intercept + 6 D × lag. Negative slopes are retained and flagged, rather than converted to positive values.

In [ ]:
available = {name: data for name, data in window["species"].items() if "msd_angstrom2" in data}
if available:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
    fit_rows = []
    for name, data in available.items():
        lag = data["lag_ps"]
        axes[0].plot(lag, data["msd_angstrom2"], label=f"{name} ({data['n_molecules']} molecules)")
        axes[1].plot(lag, data["n_origins"], label=name)
        if DIFFUSION_FIT_PS is not None:
            lo, hi = DIFFUSION_FIT_PS
            if not 0 <= lo < hi <= lag[-1]:
                raise ValueError("Diffusion fit must lie within the plotted lag range")
            fit = md.get_molecular_diffusion_coefficient(data, fit_start_ps=lo, fit_end_ps=hi)
            fit_rows.append(fit)
            x = np.array([lo, hi])
            axes[0].plot(x, fit["intercept_angstrom2"] + fit["slope_angstrom2_per_ps"]*x, "--")
            if fit["diffusion_cm2_per_s"] <= 0:
                print(name, ": nonpositive fitted slope is not a physical diffusion estimate.")
    axes[0].set(xlabel="Lag (ps)", ylabel="Molecular COM MSD (angstrom^2)")
    axes[1].set(xlabel="Lag (ps)", ylabel="Number of time origins")
    for ax in axes:
        ax.legend(); ax.grid(alpha=.2)
    finish(fig, "molecular_msd")
    if fit_rows:
        display(pd.DataFrame(fit_rows))
else:
    print("Molecular MSD unavailable for this cell; see notices above.")

## Molecular velocity and net-force correlations

The velocity is each molecule's mass-weighted COM velocity; force is the sum of its atomic forces. Curves are dot-product averages over molecules and time origins, with no mean subtraction. The figures normalize by lag zero where nonzero. Net MD force is not an environment-only external force.

The available time resolution is the retained frame interval. A rapidly decaying velocity correlation may be unresolved even if the trajectory is long.

In [ ]:
for key, title in (("vacf", "Molecular COM velocity correlation"), ("force_acf", "Molecular net-force correlation")):
    available = {name: data for name, data in window["species"].items() if key in data}
    if not available:
        print(title, ": required dump columns are absent.")
        continue
    fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
    for name, data in available.items():
        correlation = data[key]
        if correlation[0] <= 0:
            print(name, ": zero signal; normalized correlation omitted.")
            continue
        ax.plot(data["lag_ps"], correlation/correlation[0], label=name)
    ax.set(title=title, xlabel="Lag (ps)", ylabel="C(lag) / C(0)")
    ax.grid(alpha=.2); ax.legend()
    finish(fig, key)

## Optional candidate-pair persistence

Enable `COMPUTE_PAIRS` and rerun analysis to obtain counts and lifetimes for the existing CPD/dienophile screen. This can be much slower than the other metrics. Pair identities use persistent atom IDs.

Lifetime records describe continuous detections at the retained sampling interval. One-frame detections have zero observed span, not known zero physical lifetime. Records touching either analysis-window boundary are censored; their full lifetime is unknown. Striding can miss interruptions and merge apparently continuous detections. These are candidate encounters, not reactions.

In [ ]:
if result["pairs_by_frame"] is not None:
    selected_pairs = [p for p, keep in zip(result["pairs_by_frame"], window["frame_mask"]) if keep]
    selected_times = frames.time_ps.to_numpy()[window["frame_mask"]]
    lifetimes = md.get_pair_lifetime(selected_pairs, selected_times)
    lifetime_table = pd.DataFrame([asdict(event) for event in lifetimes])
    display(lifetime_table.head(20))
    if len(lifetime_table):
        complete = lifetime_table[~lifetime_table.left_censored & ~lifetime_table.right_censored]
        fig, ax = plt.subplots(figsize=(8, 4), layout="constrained")
        ax.hist(complete.observed_span_ps, bins=25)
        ax.set(xlabel="Observed continuous pair span (ps)", ylabel="Uncensored detections")
        finish(fig, "pair_lifetimes")
        print("Boundary-censored records:", len(lifetime_table)-len(complete))
else:
    print("Pair analysis disabled. Enable COMPUTE_PAIRS to include it.")